In [14]:
# ============================================================================
# BLOCK 5: TOPIC MODELING - ROMANIAN
# Input: 04_sentiment_data_ro.pkl
# Output: 05_topics_data_ro.pkl
# Runtime: ~15-20 minutes on CPU
# ============================================================================
%run 00_setup_and_config.ipynb

✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 03:03:09
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_analysis.ipynb

Checkpoint directory: c:\Users\andre\OneDrive\Des

In [15]:
# ============================================================================
# CELL 2: ROMANIAN STOPWORDS FOR BERTOPIC
# ============================================================================

STOPWORDS_RO = [
    'și', 'în', 'cu', 'din', 'pe', 'că', 'la', 'este', 'de', 'nu', 'se',
    'un', 'o', 'lui', 'ei', 'lor', 'acest', 'această', 'aceste', 'acești',
    'care', 'ce', 'cine', 'unde', 'când', 'cum', 'de ce', 'foarte', 'doar',
    'mai', 'prea', 'tot', 'toți', 'toate', 'dar', 'sau', 'ori', 'dacă',
    'pentru', 'prin', 'fără', 'despre', 'asupra', 'contra', 'după', 'înainte',
    'între', 'lângă', 'sub', 'peste', 'a', 'ai', 'am', 'au', 'avea', 'fi',
    'să', 'îmi', 'îți', 'își', 'ne', 'vă', 'mi', 'ti', 'le', 'i', 'm', 't', 's',
    'the', 'and', 'of', 'in', 'to', 'a', 'is', 'it', 'for', 'on', 'with',
    'as', 'at', 'by', 'an', 'be', 'are', 'was', 'were', 'been', 'have', 'has'
]

print(f'✓ Loaded {len(STOPWORDS_RO)} Romanian stopwords')


✓ Loaded 91 Romanian stopwords


In [16]:
# Add this at the TOP of your code to force re-run
import os

CHECKPOINT_FILE = PROCESSED_DIR / '05_topics_data_ro.pkl'
MODEL_FILE = PROCESSED_DIR / 'bertopic_model_ro'

# DELETE OLD CHECKPOINTS
if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()
    print(f'✓ Deleted old checkpoint: {CHECKPOINT_FILE}')

if MODEL_FILE.exists():
    import shutil
    shutil.rmtree(MODEL_FILE)
    print(f'✓ Deleted old model: {MODEL_FILE}')

# Now force the else branch to run


✓ Deleted old checkpoint: data\processed\05_topics_data_ro.pkl


NotADirectoryError: [WinError 267] The directory name is invalid: 'data\\processed\\bertopic_model_ro'

In [ ]:
print('='*70)
print('BLOCK 5: TOPIC MODELING - ROMANIAN')
print('='*70)

if check_checkpoint_exists('ro', '05_topics_data'):
    df_ro = load_checkpoint('ro', '05_topics_data')
    topic_model = None
    print('✓ Loading from topics checkpoint')
else:
    df_ro = load_checkpoint('ro', '04_sentiment_data')
    if df_ro is None:
        raise FileNotFoundError('Run 03_sentiment_analysis_ro.ipynb first')

print(f'\nRunning BERTopic on {len(df_ro):,} Romanian comments...')

# Load embedding model
print('\nLoading embedding model...')
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_model = embedding_model.to('cpu')
print('✓ Embedding model loaded')

# ============================================================================
# CELL 3: Run BERTopic (WORKING VERSION WITH PROPER STOPWORDS)
# ============================================================================
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
import gc

gc.collect()

if not check_checkpoint_exists('ro', '05_topics_data'):
    print('\n' + '='*70)
    print('BERTOPIC WITH AGGRESSIVE STOPWORD FILTERING')
    print('='*70)
    
    # EXTENDED Romanian stopwords - this is the KEY fix
    STOPWORDS_RO = [
        # Most common Romanian function words
        'și', 'în', 'cu', 'din', 'pe', 'că', 'la', 'este', 'de', 'nu', 'se',
        'un', 'o', 'lui', 'ei', 'lor', 'acest', 'această', 'aceste', 'acești',
        'care', 'ce', 'cine', 'unde', 'când', 'cum', 'foarte', 'doar',
        'mai', 'prea', 'tot', 'toți', 'toate', 'dar', 'sau', 'ori', 'dacă',
        'pentru', 'prin', 'fără', 'despre', 'asupra', 'contra', 'după', 'înainte',
        'între', 'lângă', 'sub', 'peste', 'a', 'ai', 'am', 'au', 'avea', 'fi',
        'să', 'îmi', 'îți', 'își', 'ne', 'vă', 'mi', 'ti', 'le', 'i', 'm', 't', 's',
        # Additional common words
        'fi', 'fost', 'fiind', 'aveam', 'aveai', 'avea', 'aveam', 'aveau',
        'sunt', 'sunteți', 'suntem', 'era', 'erau', 'eram', 'fii', 'fie',
        'pot', 'poate', 'putem', 'putea', 'vor', 'va', 'vei', 'vom',
        'acesta', 'acesta', 'acelui', 'acelea', 'acel', 'acea',
        'meu', 'mea', 'mei', 'mele', 'tău', 'ta', 'tăi', 'tale',
        'său', 'sa', 'săi', 'sale', 'nostru', 'noastră', 'noștri', 'noastre',
        'vostru', 'voastră', 'voștri', 'voastre',
        'alt', 'alta', 'alți', 'alte', 'același', 'aceeași',
        'niște', 'niște', 'câțiva', 'câteva',
        'abia', 'apoi', 'aici', 'acolo', 'atunci', 'acum', 'mereu', 'niciodată',
        'deja', 'doar', 'chiar', 'oare', 'parcă', 'cam', 'totuși', 'însă',
        'deci', 'astfel', 'așa', 'asa', 'cum', 'căci', 'că', 'ca',
        # English stopwords (appear in Romanian text too)
        'the', 'and', 'of', 'in', 'to', 'a', 'is', 'it', 'for', 'on', 'with',
        'as', 'at', 'by', 'an', 'be', 'are', 'was', 'were', 'been', 'have', 'has',
        'that', 'this', 'will', 'would', 'could', 'should', 'may', 'might',
        # Platform/video related
        'video', 'comment', 'comentariu', 'reply', 'răspuns', 'like', 'distribuire',
        'algoritm', 'algorithm', 'youtube', 'recorder', 'mulțumim', 'multumim'
    ]
    
    print(f'✓ Loaded {len(STOPWORDS_RO)} Romanian stopwords')
    
    # Custom vectorizer - THIS IS THE KEY
    vectorizer_model = CountVectorizer(
        stop_words=STOPWORDS_RO,
        ngram_range=(2, 4),           # ONLY phrases (no single words)
        min_df=10,                    # Word must appear in 10+ documents
        max_df=0.7,                   # Word can't appear in >70% of documents
        lowercase=True,
        token_pattern=r'(?u)\b\w[\w-]*\w\b|\w+\b'  # Keep words with hyphens
    )
    
    topic_model = BERTopic(
        language='multilingual',
        embedding_model=embedding_model,
        min_topic_size=15,            # Lower to capture more topics
        nr_topics=8,                  # Target ~8 topics
        calculate_probabilities=True,
        verbose=True,
        vectorizer_model=vectorizer_model,  # KEY: Custom vectorizer
        representation_model=KeyBERTInspired()
    )
    
    print('\nFitting BERTopic model...')
    
    # Get texts
    texts = df_ro['clean_text'].fillna('').astype(str).tolist()
    
    # Filter very short texts (less than 3 words)
    texts_filtered = []
    indices_kept = []
    for i, t in enumerate(texts):
        if len(t.split()) >= 3:
            texts_filtered.append(t)
            indices_kept.append(i)
    
    print(f'  Original texts: {len(texts):,}')
    print(f'  After filtering: {len(texts_filtered):,}')
    print(f'  Removed: {len(texts) - len(texts_filtered):,} short texts')
    
    # Fit model
    topics, probs = topic_model.fit_transform(texts_filtered)
    
    # Map back to original dataframe
    df_ro['topic'] = -1
    df_ro['topic_probability'] = 0.0
    
    for idx, (topic, prob) in zip(indices_kept, zip(topics, probs)):
        df_ro.loc[idx, 'topic'] = topic
        df_ro.loc[idx, 'topic_probability'] = max(prob)
    
    # Save
    topic_model.save(PROCESSED_DIR / 'bertopic_model_ro')
    print(f'✓ Topic model saved')
    
    # QUALITY CHECK
    print('\n' + '='*70)
    print('TOPIC QUALITY REPORT')
    print('='*70)
    
    topic_info = topic_model.get_topic_info()
    display(topic_info)
    
    # Count outliers
    outliers = (df_ro['topic'] == -1).sum()
    outlier_pct = 100 * outliers / len(df_ro)
    print(f'\n⚠️  OUTLIERS (Topic -1): {outliers:,} comments ({outlier_pct:.1f}%)')
    
    if outlier_pct > 30:
        print('   ⚠️  WARNING: High outlier rate - consider reducing min_topic_size')
    else:
        print('   ✓ Outlier rate acceptable')
    
    # Show topic words
    print('\n--- TOP 5 WORDS PER TOPIC ---')
    for topic_id in range(len(topic_info)):
        if topic_id == -1:
            continue
        topic_words = topic_model.get_topic(topic_id)
        words = [w for w, s in topic_words[:5]]
        count = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]
        print(f"Topic {topic_id} ({count:,} comments): {', '.join(words)}")
    
    # Save checkpoint
    save_checkpoint(df_ro, 'ro', '05_topics_data')
    update_pipeline_status('ro', 5, 'completed')
    
else:
    topic_model = BERTopic.load(PROCESSED_DIR / 'bertopic_model_ro')
    print('✓ Topic model loaded from checkpoint')



# Topic summary
print('\n' + '='*70)
print('TOPIC SUMMARY - ROMANIAN')
print('='*70)

topic_info = topic_model.get_topic_info()
display(topic_info)

topic_info.to_csv(OUTPUT_DIR / 'romanian' / 'ro_topics_summary.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'romanian' / 'ro_topics_summary.csv'}")

# Visualization
print('\nGenerating topic visualization...')

fig, ax = plt.subplots(figsize=(14, 8))
topic_sizes = topic_model.get_topic_info().sort_values('Count', ascending=False)
colors = plt.cm.Set3(np.linspace(0, 1, len(topic_sizes)))

ax.barh(range(len(topic_sizes)), topic_sizes['Count'].values, color=colors)
ax.set_yticks(range(len(topic_sizes)))
ax.set_yticklabels([f"Topic {i}: {rep[:50]}..." 
                    for i, rep in zip(topic_sizes['Topic'], topic_sizes['Representation'])])
ax.set_xlabel('Number of Comments', fontsize=12)
ax.set_title(f'Topic Distribution - Romanian (n={len(df_ro):,})', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
save_figure(fig, 'ro_topics_overview.png', 'ro', 'visualizations')

save_checkpoint(df_ro, 'ro', '05_topics_data')
update_pipeline_status('ro', 5, 'completed')

print('\n' + '='*70)
print('✓ BLOCK 5 COMPLETE - TOPIC MODELING DONE')
print('='*70)


BLOCK 5: TOPIC MODELING - ROMANIAN
✓ Loading checkpoint: 05_topics_data_ro.pkl
✓ Loading from topics checkpoint

Running BERTopic on 6,600 Romanian comments...

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2547.37it/s]


✓ Embedding model loaded
✓ Topic model loaded from checkpoint

TOPIC SUMMARY - ROMANIAN


,Topic,Count,Name,Representation,Representative_Docs
0,-1,3034,-1_de_nu_si_ca,"[de, nu, si, ca, sa, la, cu, pe, in, și]","[Pentru cei care zic ca Simion este antisistem, la fel si Georgescu, iar ei sunt oglinda celor m..."
1,0,1839,0_de_nu_sa_si,"[de, nu, sa, si, cu, ca, la, și, in, care]",[Am urmărit cu atenție toți pe cei interviati de voi. Am plecat din 1995 în afara și de când am ...
2,1,904,1_recorder_de_nu_sa,"[recorder, de, nu, sa, si, la, cu, simion, mulțumim, pentru]","[Multumim Recorder!, ATI INTREBAT OAMENI PE CINE VOR VOTA FIX IN BUCURESTI UNDE NCUSOR DAN ARE M..."
3,2,509,2_nu_de_si_ca,"[nu, de, si, ca, sa, ce, la, mai, se, cu]","[""Dintre doua rele"" - Ce rau a facut Nicusor?! Chiar vreau sa stiu, ca daca intreb de Simion put..."
4,3,185,3_video_de_la_recorder,"[video, de, la, recorder, acest, și, nu, am, pe, sa]","[Ar fi folositor daca toti votantii GS ar vedea acest video, pentru ca e usor sa fii spalat pe c..."
5,4,57,4_algoritm_comentariu_pentru_ajunga,"[algoritm, comentariu, pentru, ajunga, pt, videoul, algoritmul, sa, hai, la]","[Comentariu pentru algoritm, Comentariu pentru algoritm, Comentariu pentru algoritm]"
6,5,56,5_psd_aur_de_in,"[psd, aur, de, in, cu, pnl, sa, nu, au, ca]","[AUR+PSD=, AUR = PSD, Simion = psd]"
7,6,16,6_trist_plâns_sperante_fara,"[trist, plâns, sperante, fara, speranta, bucuria, carusel, stress, morti, release]","[Trist!, trist, Cat de trist e sa vad oameni tristi,dezamagiti, fara sperante.... Dar ma intreb ..."



✓ Saved: outputs\romanian\ro_topics_summary.csv

Generating topic visualization...
✓ Saved: outputs\ro\visualizations\ro_topics_overview.png
✓ Checkpoint saved: 05_topics_data_ro.pkl
✓ Pipeline status updated: ro - Block 5 - completed

✓ BLOCK 5 COMPLETE - TOPIC MODELING DONE


In [ ]:
# Diagnostic: Show full topic info
topic_info = topic_model.get_topic_info()
print(topic_info.to_string())

# Show top 10 words for each topic
print('\n--- TOP 10 WORDS PER TOPIC ---')
for topic_id in range(min(8, len(topic_info))):
    topic_words = topic_model.get_topic(topic_id)
    print(f'\nTopic {topic_id}:')
    for word, score in topic_words[:10]:
        print(f'  {word}: {score:.4f}')

   Topic  Count                                 Name                                                                      Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

TypeError: 'bool' object is not subscriptable